# Integre seu próprio MCP Server ao AgentCore Gateway

## Visão Geral

Em grandes empresas com milhares de equipes de desenvolvimento, cada uma executando múltiplos MCP servers com diversas ferramentas, gerenciar e descobrir ferramentas se torna um desafio crítico:

* **Descoberta e Compartilhamento de Ferramentas**: As equipes têm dificuldade em descobrir e consumir ferramentas em toda a organização. Cada equipe deve manter gateways separados? Como compartilhar URLs de gateway e gerenciar registros centrais sem criar sobrecarga operacional?
* **Gerenciamento de Gateway**: Manter gateways separados para cada MCP server rapidamente se torna ingerenciável em escala.
* **Complexidade de Autenticação**: Gerenciar autenticação e autorização em múltiplos MCP servers se torna cada vez mais complexo, especialmente com dados empresariais sensíveis.
* **Sobrecarga de Manutenção**: Acompanhar as atualizações da especificação MCP requer retrabalho e testes contínuos em todas as implementações.

O AgentCore Gateway agora suporta a integração de implementações de MCP server pré-existentes como targets, além de ferramentas Lambda e OpenAPI. Este tutorial demonstra o conceito usando um MCP server simples no AgentCore Runtime para esta implementação.

![Diagrama](images/mcp-server-target.png)

Uma vez que o AgentCore Gateway esteja integrado com o MCP server como target, buscaremos ferramentas através do nosso MCP server integrado, e então usaremos um Strands Agent para demonstrar as funcionalidades de listar ferramentas e invocar ferramentas. Mais detalhes sobre cada um desses fluxos podem ser encontrados abaixo:

### Comportamento de busca de ferramentas para MCP Targets

A capacidade de busca no AgentCore Gateway permite a descoberta semântica de ferramentas em todos os tipos de target, incluindo MCP targets. Para MCP targets, a funcionalidade de busca opera em definições de ferramentas normalizadas que foram capturadas e indexadas durante operações de sincronização, proporcionando busca semântica eficiente sem comunicação em tempo real com o MCP server. Quando as definições de ferramentas são sincronizadas a partir de um MCP target, o Gateway gera automaticamente embeddings para o nome, descrição e descrições de parâmetros de cada ferramenta. Esses embeddings são armazenados junto com as definições de ferramentas normalizadas, permitindo busca semântica que compreende a intenção e o contexto das consultas de busca. Diferente da correspondência tradicional por palavras-chave, isso permite que agentes descubram ferramentas relevantes mesmo quando a terminologia exata não corresponde.

![Busca](images/mcp-server-search-tool.png)

### Comportamento de ListTools para MCP Targets

A operação ListTools no AgentCore Gateway fornece acesso às definições de ferramentas previamente sincronizadas a partir de MCP targets, seguindo uma abordagem cache-first que prioriza desempenho e confiabilidade. Diferente dos targets tradicionais OpenAPI ou Lambda onde as definições de ferramentas são definidas estaticamente, as ferramentas de MCP target são descobertas e armazenadas em cache através de operações de sincronização. Quando um cliente chama ListTools, o Gateway recupera as definições de ferramentas do seu armazenamento persistente em vez de fazer chamadas em tempo real ao MCP server. Essas definições foram previamente preenchidas através de sincronização implícita durante a criação/atualização do target ou através de chamadas explícitas à API SynchronizeGateway. A operação retorna uma lista paginada de definições de ferramentas normalizadas.

![Listar](images/mcp-server-list-tools.png)

### Comportamento de InvokeTool (tools/call) para MCP Targets

A operação InvokeTool para MCP targets lida com a execução real de ferramentas descobertas através de ListTools, gerenciando a comunicação em tempo real com o MCP server de destino. Diferente da operação ListTools baseada em cache, tools/call requer comunicação ativa com o MCP server, introduzindo requisitos específicos de autenticação, gerenciamento de sessão e tratamento de erros. Quando uma requisição tools/call chega, o Gateway primeiro valida se a ferramenta existe em suas definições sincronizadas. Para MCP targets, o Gateway realiza uma chamada inicial 'initialize' para estabelecer uma sessão com o MCP server. Se o target estiver configurado com credenciais OAuth, o Gateway recupera credenciais atualizadas do AgentCore Identity antes de fazer a chamada de inicialização. Isso garante que, mesmo que ListTools tenha retornado ferramentas em cache com credenciais expiradas, a invocação real utilize autenticação válida.

![Invocar](images/mcp-server-invoke-tool.png)


### Detalhes do Tutorial


| Informação           | Detalhes                                                  |
|:---------------------|:----------------------------------------------------------|
| Tipo de tutorial     | Interativo                                                |
| Componentes AgentCore| AgentCore Gateway, AgentCore Identity                     |
| Framework de Agentes | Strands Agents                                            |
| Tipo de Gateway Target| MCP server                                               |
| Agente               | Strands                                                   |
| Inbound Auth IdP| Amazon Cognito, mas pode usar outros                     |
| Outbound Auth        | Amazon Cognito, mas pode usar outros                      |
| Modelo LLM           | Anthropic Claude Sonnet 4                                 |
| Componentes do tutorial| Criação do AgentCore Gateway e Invocação do AgentCore Gateway |
| Vertical do tutorial | Cross-vertical                                            |
| Complexidade do exemplo| Fácil                                                   |
| SDK utilizado        | boto3                                                     |

## Arquitetura do Tutorial

Este tutorial serve como um exemplo prático do desafio empresarial mais amplo: **Como integrar MCP servers existentes em uma arquitetura de Gateway centralizada.**

## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook (kernel Python)
* uv
* Credenciais AWS
* Amazon Cognito

In [ ]:
# Install from the requirements file or pyproject.toml file in current directory
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import os
# Set AWS credentials if not using SageMaker notebook
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1')

In [ ]:
# Import utils
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

# Setup logging 
import logging

# Configure logging for notebook environment
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

# Set specific logger levels
logging.getLogger("strands").setLevel(logging.INFO)


## Criar uma IAM role para o Gateway assumir

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-mcpgateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

## Criar Amazon Cognito Pool para inbound auth no Gateway

In [ ]:
# Creating Cognito User Pool 
import os
import boto3

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
    "ScopeDescription": "Scope for invoking the agentcore gateway"},
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names)

print(f"Client ID: {gw_client_id}")

# Get discovery URL
gw_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration'
print(gw_cognito_discovery_url)

## Criar Amazon Cognito Pool para inbound auth no Runtime
Esta seção destaca a capacidade do gateway de ter uma inbound auth separada dos seus sistemas de destino. Isso permite que agentes acessem ferramentas que utilizam múltiplos provedores de identidade através de uma única interface.

In [ ]:
# Creating Cognito User Pool
import os
import boto3

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-runtime-pool"
RESOURCE_SERVER_ID = "sample-agentcore-runtime-id"
RESOURCE_SERVER_NAME = "sample-agentcore-runtime-name"
CLIENT_NAME = "sample-agentcore-runtime-client"
SCOPES = [
    {"ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
    "ScopeDescription": "Scope for invoking the agentcore gateway"},
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
runtimeScopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
runtime_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {runtime_user_pool_id}")

utils.get_or_create_resource_server(cognito, runtime_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

runtime_client_id, runtime_client_secret = utils.get_or_create_m2m_client(cognito, runtime_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names)

print(f"Client ID: {runtime_client_id}")

# Get discovery URL
runtime_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{runtime_user_pool_id}/.well-known/openid-configuration'
print(runtime_cognito_discovery_url)

## Criar o Gateway

In [ ]:
# CreateGateway with Cognito authorizer. Use the Cognito user pool created in the previous step
import boto3
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [gw_client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": gw_cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='ac-gateway-mcp-server',
    roleArn=agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway
    protocolType='MCP',
    protocolConfiguration={
        'mcp': {
            'supportedVersions': ['2025-03-26'],
            'searchType': 'SEMANTIC'
        }
    },
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config,
    description='AgentCore Gateway with MCP Server target'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

## Criar um MCP Server de exemplo e hospedá-lo no Runtime

Você pode substituir o código abaixo pelo seu MCP server que já pode existir. Esta seção segue o exemplo do tutorial [Hospedando MCP Server no Amazon Bedrock AgentCore Runtime](https://github.com/awslabs/amazon-bedrock-agentcore-samples/blob/main/01-tutorials/01-AgentCore-runtime/02-hosting-MCP-server/hosting_mcp_server.ipynb).

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

Em seguida, configure o ambiente do AgentCore Runtime.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

required_files = ['mcp_server.py', 'requirements.txt']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")
agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            runtime_client_id
        ],
        "discoveryUrl": runtime_cognito_discovery_url
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name="ac_gateway_mcp_server"
)
print("Configuration completed ✓")

Então, faça o deploy no AgentCore Runtime.

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()

agent_arn = launch_result.agent_arn
agent_id = launch_result.agent_id

encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')

agent_url = f'https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT'
print("Launch completed ✓")
print(f"Agent ARN: {agent_arn}")
print(f"Agent ID: {agent_id}")

## Criar um MCP server como target para o AgentCore Gateway

### Configurar Outbound Auth

Primeiro, criaremos um AgentCore Identity Resource Credential Provider para que nosso AgentCore Gateway use como outbound auth para nosso MCP server no AgentCore Runtime.

In [ ]:
import boto3
identity_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name="ac-gateway-mcp-server-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        'customOauth2ProviderConfig': {
            'oauthDiscovery': {
                'discoveryUrl': runtime_cognito_discovery_url,
            },
            'clientId': runtime_client_id,
            'clientSecret': runtime_client_secret
        }
    }
)
cognito_provider_arn = cognito_provider['credentialProviderArn']
print(cognito_provider_arn)

### Criar o Gateway Target

In [ ]:
import boto3
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
create_gateway_target_response = gateway_client.create_gateway_target(
    name='mcp-server-target',
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        'mcp': {
            'mcpServer': {
                'endpoint': agent_url
            }
        }
    },
    credentialProviderConfigurations=[
        {
            'credentialProviderType': 'OAUTH',
            'credentialProvider': {
                'oauthCredentialProvider': {
                    'providerArn': cognito_provider_arn,
                    'scopes': [
                        runtimeScopeString
                    ]
                }
            }
        },
    ]
)
print(create_gateway_target_response)

#### Verificar se o Gateway target existe e está READY

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(list_targets_response)

#### Solicitar o access token do Amazon Cognito para inbound auth

In [ ]:
print("Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propagation completes")
token_response = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
token = token_response["access_token"]
print("Token response:", token)

## Buscar ferramentas do MCP Server através do Gateway

In [ ]:
import requests
import json

def search_tools(gateway_url, access_token, query):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}"
    }

    payload = {
        "jsonrpc": "2.0",
        "id": "search-tools-request",
        "method": "tools/call",
        "params": {
            "name": "x_amz_bedrock_agentcore_search",
            "arguments": {
                "query": query
            }
        }
    }

    response = requests.post(gateway_url, headers=headers, json=payload)
    return response.json()

# Example usage
token_response = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
access_token = token_response['access_token']
results = search_tools(gatewayURL, access_token, "orders")
print(json.dumps(results, indent=2))

Para mais exemplos relevantes sobre a funcionalidade de busca do AgentCore Gateway, veja [03-search-tools](../03-search-tools).

## Usar um agente para operar em pedidos através do Gateway

In [ ]:
import json

import requests
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def get_token():
    token = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
    return token['access_token']


def create_streamable_http_transport():
    return streamablehttp_client(
        gatewayURL, headers={"Authorization": f"Bearer {get_token()}"}
    )


client = MCPClient(create_streamable_http_transport)

## The IAM group/user/ configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", # may need to update model_id depending on region
    temperature=0.7,
    max_tokens=500,  # Limit response length
)

with client:
    # Call the listTools
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(
        model=yourmodel, tools=tools
    )  ## you can replace with any model you like
    # Invoke the agent with the sample prompt. This will only invoke MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi, can you list all tools available to you")
    # Simplified prompt and error handling
    result = agent("Update order 123")

## Limpeza
Recursos adicionais também são criados, como IAM role, IAM Policies, Credentials provider, funções AWS Lambda, Cognito user pools, buckets S3, que você pode precisar excluir manualmente como parte da limpeza. Isso depende do exemplo que você executar.

### NOTA: se você vai prosseguir para o próximo notebook, [02-mcp-target-synchronization](02-mcp-target-synchronization.ipynb), sugerimos manter esses recursos para o próximo tutorial.

### Excluir o Gateway (Opcional)

In [ ]:
utils.delete_gateway(gateway_client, gatewayID)

### Excluir o Identity Provider (Opcional)

In [ ]:
identity_client.delete_oauth2_credential_provider(name='ac-gateway-mcp-server-identity')

### Excluir o Runtime (Opcional)

In [ ]:
import boto3
runtime_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
runtime_client.delete_agent_runtime(agentRuntimeId=agent_id)